#### Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema types and methods for enforcing structured output. 

#### Pydantic :

#### Pydantic module provides the richest feature set with field validation, description and nested structures.

In [16]:
## integrate with Groq
import os
from langchain.chat_models import init_chat_model
from langchain_groq import ChatGroq # another way of using groq

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x117026850>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x117026490>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [14]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year the movie was released")
    director:str=Field(description="Director of the movie")
    ratings:float=Field(description="The movies ratings out of 10")

In [15]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x1161bde50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1161be850>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'Director of the movie', 'type': 'string'}, 'ratings': {'description': 'The movies ratings out of 10', 'type': 

In [ ]:
# unstructured output
model.invoke("Provide me the details of the movie Inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know. Inception is a science fiction action film directed by Christopher Nolan. It was released in 2010, I think. The main cast includes Leonardo DiCaprio, who plays the lead role. The story is about entering dreams to plant ideas, right? The term for that is "inception," which is the title of the movie.\n\nI remember the plot involves Dom Cobb, the main character, who is a thief who steals information by infiltrating the subconscious. The team of specialists he assembles to perform an inception on a target. The target is a businessman named Robert Fischer, and the goal is to plant an idea in his mind to make him dissolve his father\'s business empire. \n\nThere\'s a lot of complexity in the narrative with multiple layers of dreams. Each layer has different time dilation effects, so time moves slower the deeper you go. The deeper layers are more dangerous because t

In [7]:
## Strucutred output
model_with_structure.invoke("Provide me the details of the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', ratings=8.8)

In [ ]:
## Message output alongside  parsed structure

from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details. """
    title:str=Field(...,description="The title of the movie") # ... means optional field
    year:int=Field(...,description="The year the movie was released")
    director:str=Field(...,description="Director of the movie")
    ratings:float=Field(...,description="The movies ratings out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True) # gives both raw and structured message
response = model_with_structure.invoke("provide details about movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me see. I need to use the Movie function they provided. The required parameters are title, year, director, and ratings. First, I should confirm the title is "Inception". The director is Christopher Nolan, right? I think the release year was 2010. As for ratings, I remember it has a high rating on IMDb, maybe around 8.8. Let me double-check those details to make sure. Yeah, director is definitely Nolan, year 2010, and the ratings are 8.8. Alright, that\'s all the required info. Time to structure the JSON with those parameters.\n', 'tool_calls': [{'id': '0wkrzwwfk', 'function': {'arguments': '{"director":"Christopher Nolan","ratings":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 193, 'prompt_tokens': 229, 'total_tokens': 422, 'completion_time': 0.355231352, 'comp

In [12]:
from groq.resources import models
from groq.resources import models
## Nested Structure

from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor] # nested 
    genres:list[str]
    budget:float|None=Field(None, description="Budget in million USD")
model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("provide details about movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Balthazar')], genres=['Science Fiction', 'Action', 'Thriller'], budget=160.0)

#### TypedDict :  provides a simpler alternative using python built-in typing ideal when you dont need runtime validation


In [19]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details. """
    title: Annotated[str, ...,"The title of the movie"] # ... means optional field
    year:Annotated[int, ...,"The year the movie was released"]
    director:Annotated[str, ...,"Director of the movie"]
    ratings:Annotated[float, ...,"The movies ratings out of 10"]

model_withtypeddict = model.with_structured_output(MovieDict)
response = model_withtypeddict.invoke("Please provode the details of movie avengers")
response


{'director': 'Joss Whedon', 'ratings': 8, 'title': 'Avengers', 'year': 2012}

In [20]:
# input vlidation will not happen in TYPEDDICT unlike pydantic
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor] # nested 
    genres:list[str]
    budget:float|None=Field(None, description="Budget in million USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("provide details about movie Inception")
response

{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Ellen Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Jet'}],
 'genres': ['Action', 'Sci-Fi', 'Thriller'],
 'title': 'Inception',
 'year': 2010}

#### DataClasses :  A data class typically contains a data , although thre are no restrictions. You create this using @dataclass decorator.



In [21]:
import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [22]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """contact information of a person"""
    name:str=Field(description="The name of the person")
    phone:str=Field(description="The phone number of the person")
    email:str=Field(description="Email of the person")

agent = create_agent(
    model = "gpt-5",
    response_format = ContactInfo # auto-selects ProviderStrategy

) 
result = agent.invoke(
    {
      "messages":   [{"role":"user","content":"Extract contact info from : john Doe, john@example.com, (123) 456-7687"}]
    }
)
result
#print(result["strucrured_response"])

{'messages': [HumanMessage(content='Extract contact info from : john Doe, john@example.com, (123) 456-7687', additional_kwargs={}, response_metadata={}, id='11244070-1618-4701-9041-00fef1a36f24'),
  AIMessage(content='{"name":"john Doe","phone":"(123) 456-7687","email":"john@example.com"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 804, 'prompt_tokens': 202, 'total_tokens': 1006, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DnrGlvpBEOWRwOP7aO9xgXNZsHtMi', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e9e79-4bec-7a23-a766-072720d833ba-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 202, 'outp

In [24]:
print(result["structured_response"])

name='john Doe' phone='(123) 456-7687' email='john@example.com'


In [ ]:
## Using the same with TypedDict

from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """contact information of a person"""
    name:str#The name of the person
    phone:str# The phone number of the person
    email:str #"Email of the person

agent = create_agent(
    model = "gpt-5",
    response_format = ContactInfo # auto-selects ProviderStrategy

) 
result = agent.invoke(
    {
      "messages":   [{"role":"user","content":"Extract contact info from : john Doe, john@example.com, (123) 456-7687"}]
    }
)
print(result["structured_response"]) ## O/P in dictionary Object

{'name': 'john Doe', 'phone': '(123) 456-7687', 'email': 'john@example.com'}


In [ ]:
## with the help of data class

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """contact information of a person"""
    name:str#The name of the person
    phone:str# The phone number of the person
    email:str #"Email of the person

agent = create_agent(
    model = "gpt-5",
    response_format = ContactInfo # auto-selects ProviderStrategy

) 

result = agent.invoke(
    {
      "messages":   [{"role":"user","content":"Extract contact info from : john Doe, john@example.com, (123) 456-7687"}]
    }
)
print(result["structured_response"]) ## O/P using dataclass 

ContactInfo(name='john Doe', phone='(123) 456-7687', email='john@example.com')
